In [23]:
import numpy as np

rng = np.random.default_rng(42)


class Function:
    def __init__(self) -> None:
        def quadratic(x, a):
            return ((x - a) ** 2) / 2

        def d_quadratic(x, a):
            return x - a

        def d_sigmoid(s_result):
            return s_result * (1 - s_result)

        def cross_entropy(x, a):
            return a * np.log(x) + (1 - a) * np.log(1 - x)

        def sigmoid(x):
            return 1 / (1 + np.exp(-x))

        self.quadratic = quadratic
        self.d_quadratic = d_quadratic
        self.sigmoid = sigmoid
        self.d_sigmoid = d_sigmoid


func = Function()


class Cost:
    def __init__(self) -> None:
        self.quadratic = func.quadratic
        self.d_quadrat = func.d_quadratic
        pass


class Activation:
    def __init__(self) -> None:
        self.sigmoid = func.sigmoid
        self.d_sigmoid = func.d_sigmoid
        pass


cost_function = Cost()
activation_function = Activation()


class Network:
    def __init__(self, shape, batch) -> None:
        self.shape = shape
        self.layer = len(shape)
        self.batch = batch
        pass

    def initialize(self):
        self.activation = [
            np.zeros([self.shape[l], self.batch]) for l in range(self.layer)
        ]
        self.weight = [
            rng.normal(0, 1, [self.shape[l], self.shape[l + 1]])
            for l in range(self.layer - 1)
        ]
        self.d_weight = [
            np.zeros([self.shape[l], self.shape[l + 1]]) for l in range(self.layer - 1)
        ]
        self.bias = [
            rng.normal(0, 1, [self.shape[l + 1], 1]) for l in range(self.layer - 1)
        ]
        self.d_bias = [np.zeros([self.shape[l + 1], 1]) for l in range(self.layer - 1)]
        self.ds = [np.zeros([self.shape[l], self.batch]) for l in range(self.layer)]
        self.grad_b = [np.zeros([self.shape[l + 1], 1]) for l in range(self.layer - 1)]
        self.grad_w = [
            np.zeros([self.shape[l], self.shape[l + 1]]) for l in range(self.layer - 1)
        ]

    def initialize_oi(self):
        self.activation[0] = rng.normal(0, 0.5, [self.shape[0], self.batch])
        self.gt = np.zeros([self.shape[-1], self.batch])


class Propagtion:
    def __init__(
        self,
        eta,
        network,
    ) -> None:
        self.eta = eta
        self.net = network
        pass

    def Forward_Propagation(self):
        for l in range(self.net.layer - 1):
            self.net.activation[l + 1] = func.sigmoid(
                self.net.weight[l].T @ self.net.activation[l] + self.net.bias[l]
            )
            self.net.ds[l] = self.net.activation[l] * (1 - self.net.activation[l])
        pass

    def Back_Propagation(self):
        cache = self.net.activation[-1] - self.net.gt
        self.net.grad_w[-1] = cache @ self.net.activation[-1].T
        self.net.grad_w[-1] = np.sum(cache, axis=1)

        for l in range(2, self.net.layer):
            print(l)
            print(f"cache {np.shape(cache)}")
            print(f" a {np.shape(self.net.activation[-1])}")
            print(f" w  {np.shape(self.net.weight[-l+1])}")
            print(f" ds  {np.shape(self.net.ds[-l])}")
            cache = self.net.weight[-l + 1] @ cache * self.net.ds[-l]
            self.net.grad_b[-l] = cache
            self.net.grad_w[-l] = cache @ self.net.activation[-l + 1].T

    def Gradient_Descent(self):
        for l in range(self.net.layer - 1):
            print(l)
            print(f" w {np.shape(self.net.weight[l])}")
            print(f" gw  {np.shape(self.net.grad_w[l])}")
            print(f" b {np.shape(self.net.bias[l])}")
            print(f" gb  {np.shape(self.net.grad_b[l])}")
            self.net.weight[l] -= self.eta * self.net.grad_w[l]
            self.net.bias[l] -= self.eta * self.net.grad_b[l]
        pass


shape = [3, 2, 4]
batch = 5
eta = 0.5
epoch = 1


network = Network(shape, batch)
network.initialize()
network.initialize_oi()
propgation = Propagtion(eta, network)
propgation.Forward_Propagation()
propgation.Back_Propagation()
propgation.Gradient_Descent()


2
cache (4, 5)
 a (4, 5)
 w  (2, 4)
 ds  (2, 5)
0
 w (3, 2)
 gw  (2, 4)
 b (2, 1)
 gb  (2, 5)


ValueError: operands could not be broadcast together with shapes (3,2) (2,4) (3,2) 